In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import torch
import numpy as np

def remap_mask(mask):
    # Remaps values to 0, 1, 2... based on unique values present
    mask = mask.long() # Segmentation masks must be LongTensor
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask


import os
import glob
from torch.utils.data import Dataset
from PIL import Image

class MulticlassDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None, mask_transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 1. Load Image & Mask
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L") # Keep as L (grayscale)

        # 2. Apply Transform
        if self.transform:
            image = self.transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)

        # 3. Remap Mask (we need this because CrossEntropy requires the labels to be consecutive)
        mask = remap_mask(mask)

        return image, mask



from sklearn.model_selection import train_test_split
from torchvision import transforms
from torch.utils.data import DataLoader

# Let's assume the data is not splitted here, we will use only the training folder, then split it.
# 1. Define Paths
root_dir = os.path.join(path, "Dataset")
all_images = sorted(glob.glob(f"/root/.cache/kagglehub/datasets/mohammad2012191/q3-stage3-2026/versions/1/dataset/images/*.jpg"))
all_masks  = sorted(glob.glob(f"/root/.cache/kagglehub/datasets/mohammad2012191/q3-stage3-2026/versions/1/dataset/masks/*.png")) # Check extension!




# 2. Split Data
train_imgs, test_imgs, train_masks, test_masks = train_test_split(
    all_images, all_masks, test_size=0.2, random_state=42
)

# 3. Define Transform
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()         # ToTensor does two things: Convert to tensor + scaling (divide by 255)
])
transform_mask = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.PILToTensor(),
    # PILToTensor does one thing: Convert to tensor only (mask should not be scaled!!)
])

# 4. Create Datasets
train_dataset = MulticlassDataset(train_imgs, train_masks, transform=transform, mask_transform=transform_mask)
test_dataset  = MulticlassDataset(test_imgs,  test_masks,  transform=transform, mask_transform=transform_mask)

# 5. Check Output



In [ ]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)


In [ ]:
# TO DO
import matplotlib.pyplot as plt

# Display some images with their masks
for i in range(10,13):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img.permute(1, 2, 0))  # Convert (C, H, W) to (H, W, C)
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.squeeze(), cmap="gray")
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")
    plt.show()

In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO
import segmentation_models_pytorch as smp

# Define U-Net Model
device = "cpu"
model = smp.Unet(
    encoder_name="efficientnet-b0",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8,  # Binary segmentation (1 output channel)
).to(device)

In [ ]:
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):                                  #because every oixel is each a backround or an person ect..
        images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)  # mask shape becomes [N, H, W]

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)    # mask shape becomes [N, H, W]

            outputs = model(images)  # Now [N, H, W]
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# Define loss function and optimizer
#criterion = nn.BCELoss() if not multiclass
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


In [ ]:
# TO DO
import matplotlib.pyplot as plt

# Function to denormalize images (We cannot show normalized images. We have to reverse normalizaion first.)
def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
    std = np.array([0.229, 0.224, 0.225])  # ImageNet std
    img = img.numpy().transpose(1, 2, 0)  # Convert to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip values to [0,1]
    return img

# Display some images with their masks
for i in range(3):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.permute(1,2,0), cmap="gray")
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")
    plt.show()
